# Crime Data Exploration — Los Angeles (2020–2025)

Análise exploratória do dataset processado `crime_2020_2025_clean.csv`.

**Objetivos:**
- Entender a estrutura e qualidade de cada coluna
- Verificar cobertura de coordenadas geográficas individuais (lat/lon)
- Explorar padrões temporais (hora, dia da semana, mês, ano)
- Analisar tipos de crime, perfil das vítimas e áreas do LAPD
- Extrair insights para o heatmap e risk_score do app Flutter

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

CRIME_PATH = Path('..') / 'data' / 'processed' / 'crime' / 'crime_2020_2025_clean.csv'

df = pd.read_csv(CRIME_PATH, low_memory=False)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f'Registros : {len(df):,}')
print(f'Período   : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Colunas   : {list(df.columns)}')

---
## 1. Estrutura e qualidade das colunas

In [ ]:
resumo = pd.DataFrame({
    'dtype'   : df.dtypes.astype(str),
    'nulos'   : df.isnull().sum(),
    'nulos_%' : (df.isnull().mean() * 100).round(1),
    'únicos'  : df.nunique(),
    'exemplo' : [str(df[c].dropna().iloc[0]) for c in df.columns],
})
print('=== RESUMO DAS COLUNAS ===')
print(resumo.to_string())

print('\n=== ESTATÍSTICAS NUMÉRICAS ===')
print(df[['vict_age', 'weapon_used_cd', 'lat', 'lon']].describe().round(4).to_string())

---
## 2. Coordenadas geográficas individuais (lat / lon)

In [ ]:
total     = len(df)
invalidos = ((df['lat'] == 0) | (df['lon'] == 0) |
             df['lat'].isna() | df['lon'].isna()).sum()
validos   = total - invalidos

print('=== COBERTURA DE COORDENADAS ===')
print(f'Total de registros             : {total:,}')
print(f'Com lat/lon válidos            : {validos:,}  ({validos/total*100:.1f}%)')
print(f'Sem coordenada (nulo ou zero)  : {invalidos:,}  ({invalidos/total*100:.1f}%)')
print(f'Latitude  : {df["lat"].min():.4f} → {df["lat"].max():.4f}')
print(f'Longitude : {df["lon"].min():.4f} → {df["lon"].max():.4f}')

df_v = df[(df['lat'] != 0) & (df['lon'] != 0) &
          df['lat'].notna() & df['lon'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter — amostra 5%
samp = df_v.sample(frac=0.05, random_state=42)
axes[0].scatter(samp['lon'], samp['lat'], alpha=0.05, s=1, color='#F4821E')
axes[0].set_title(f'Pontos individuais — amostra 5%\n({len(samp):,} registros)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_aspect('equal')

# Heatmap 2D de densidade — 100% dos registros
h = axes[1].hist2d(df_v['lon'], df_v['lat'], bins=100, cmap='YlOrRd', density=True)
plt.colorbar(h[3], ax=axes[1], label='Densidade relativa')
axes[1].set_title(f'Densidade de crimes\n{len(df_v):,} registros (2020–2025)')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
axes[1].set_aspect('equal')

plt.suptitle('Distribuição geográfica dos crimes em Los Angeles', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Tipos de crime

In [ ]:
print(f'Categorias únicas de crime: {df["crm_cd_desc"].nunique()}')

top25 = df['crm_cd_desc'].value_counts().head(25)

fig, ax = plt.subplots(figsize=(12, 9))
top25.sort_values().plot(kind='barh', ax=ax, color='#F4821E')
ax.set_title('Top 25 tipos de crime — Los Angeles (2020–2025)', fontsize=13)
ax.set_xlabel('Ocorrências')
ax.bar_label(ax.containers[0], fmt='{:,.0f}', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

print('\nTop 10 em detalhe:')
print(top25.head(10).to_string())

---
## 4. Padrões temporais

In [ ]:
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['month']       = df['timestamp'].dt.month
df['year']        = df['timestamp'].dt.year

ordem_dias  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
nomes_dias  = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
nomes_meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

crime_hora = df.groupby('hour').size()
crime_dia  = df.groupby('day_of_week').size().reindex(ordem_dias)
crime_mes  = df.groupby('month').size()
crime_ano  = df.groupby('year').size()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].bar(crime_hora.index, crime_hora.values, color='#F4821E')
axes[0,0].set_title('Ocorrências por hora do dia')
axes[0,0].set_xlabel('Hora'); axes[0,0].set_ylabel('Ocorrências')
axes[0,0].set_xticks(range(0, 24))

axes[0,1].bar(range(7), crime_dia.values, color='#1E88A8')
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(nomes_dias)
axes[0,1].set_title('Ocorrências por dia da semana')
axes[0,1].set_ylabel('Ocorrências')

axes[1,0].bar(range(12), crime_mes.values, color='#4CAF50')
axes[1,0].set_xticks(range(12))
axes[1,0].set_xticklabels(nomes_meses)
axes[1,0].set_title('Ocorrências por mês (2020–2025 acumulado)')
axes[1,0].set_ylabel('Ocorrências')

axes[1,1].bar(crime_ano.index.astype(str), crime_ano.values, color='#9C27B0')
axes[1,1].set_title('Ocorrências por ano')
axes[1,1].set_ylabel('Ocorrências')
axes[1,1].bar_label(axes[1,1].containers[0], fmt='{:,.0f}', padding=3)

plt.suptitle('Padrões temporais de crime — Los Angeles', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Hora de pico  : {crime_hora.idxmax()}h  ({crime_hora.max():,} ocorrências)')
print(f'Dia de pico   : {crime_dia.idxmax()}  ({crime_dia.max():,} ocorrências)')
print(f'Mês de pico   : {crime_mes.idxmax()}  ({crime_mes.max():,} ocorrências)')

In [ ]:
# Heatmap hora × dia da semana
pivot_hd = df.pivot_table(
    values='timestamp', index='day_of_week', columns='hour', aggfunc='count'
).reindex(ordem_dias)

plt.figure(figsize=(16, 5))
sns.heatmap(pivot_hd, cmap='YlOrRd', linewidths=0.2,
            annot=False, cbar_kws={'label': 'Ocorrências'})
plt.title('Heatmap — ocorrências por hora × dia da semana (2020–2025)', fontsize=13)
plt.xlabel('Hora do dia')
plt.tight_layout()
plt.show()

# Curvas anuais sobrepostas — estabilidade interanual
crime_ano_hora = df.groupby(['year', 'hour']).size().unstack(0)

plt.figure(figsize=(13, 5))
for ano in crime_ano_hora.columns:
    plt.plot(crime_ano_hora.index, crime_ano_hora[ano], label=str(ano), linewidth=2)
plt.title('Padrão horário por ano — estabilidade interanual', fontsize=13)
plt.xlabel('Hora'); plt.ylabel('Ocorrências')
plt.xticks(range(0, 24))
plt.legend(title='Ano')
plt.tight_layout()
plt.show()

---
## 5. Áreas LAPD

In [ ]:
area_total = df.groupby('area_name').size().sort_values(ascending=False)
top5 = area_total.head(5).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Volume total por área
area_total.sort_values().plot(kind='barh', ax=axes[0], color='#9C27B0')
axes[0].set_title('Total de crimes por área LAPD (2020–2025)')
axes[0].set_xlabel('Ocorrências')

# Padrão horário das top 5 áreas
crime_hora_area = (
    df[df['area_name'].isin(top5)]
    .groupby(['area_name', 'hour']).size()
    .reset_index(name='count')
)
for area in top5:
    d = crime_hora_area[crime_hora_area['area_name'] == area]
    axes[1].plot(d['hour'], d['count'], label=area, linewidth=2)
axes[1].set_title('Padrão horário — Top 5 áreas LAPD')
axes[1].set_xlabel('Hora'); axes[1].set_ylabel('Ocorrências')
axes[1].set_xticks(range(0, 24))
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'Total de áreas: {df["area_name"].nunique()}')
print(f'Top 5: {top5}')
print()
print(area_total.to_string())

---
## 6. Perfil das vítimas e uso de arma

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sexo da vítima
sex = df['vict_sex'].value_counts(dropna=False)
sex.index = sex.index.fillna('NaN')
sex.plot(kind='bar', ax=axes[0], color='#1E88A8')
axes[0].set_title('Sexo da vítima')
axes[0].set_xlabel('vict_sex'); axes[0].set_ylabel('Ocorrências')
axes[0].tick_params(axis='x', rotation=0)
axes[0].bar_label(axes[0].containers[0], fmt='{:,.0f}', padding=3, fontsize=8)

# Distribuição de idade (excluindo age == 0)
idade = df[df['vict_age'] > 0]['vict_age']
axes[1].hist(idade, bins=50, color='#4CAF50', edgecolor='white')
axes[1].axvline(idade.median(), color='red', linestyle='--',
                label=f'Mediana: {idade.median():.0f} anos')
axes[1].set_title(f'Idade da vítima (excl. age=0)\nn={len(idade):,} de {len(df):,}')
axes[1].set_xlabel('Idade'); axes[1].set_ylabel('Ocorrências')
axes[1].legend()

# Crime com ou sem arma declarada
com_arma = df['weapon_used_cd'].notna().sum()
sem_arma = df['weapon_used_cd'].isna().sum()
axes[2].bar(['Com arma', 'Sem arma'], [com_arma, sem_arma],
            color=['#E53935', '#78909C'])
axes[2].set_title('Arma declarada no crime')
axes[2].set_ylabel('Ocorrências')
axes[2].bar_label(axes[2].containers[0], fmt='{:,.0f}', padding=3)

plt.suptitle('Perfil das vítimas e caracterização dos crimes', fontsize=13)
plt.tight_layout()
plt.show()

print(f'vict_age == 0 (sem info)  : {(df["vict_age"] == 0).sum():,}  ({(df["vict_age"] == 0).mean()*100:.1f}%)')
print(f'vict_sex nulos            : {df["vict_sex"].isna().sum():,}  ({df["vict_sex"].isna().mean()*100:.1f}%)')
print(f'weapon_used_cd nulos      : {df["weapon_used_cd"].isna().sum():,}  ({df["weapon_used_cd"].isna().mean()*100:.1f}%)  → crimes sem arma declarada')
print(f'Mediana de idade          : {idade.median():.0f} anos')

---
## 7. Conclusões para o app

### Estrutura do dataset (`crime_2020_2025_clean.csv`)

| Coluna | Tipo | Cobertura | O que é |
|---|---|---|---|
| `timestamp` | datetime64 | 100% | Data e hora exata da ocorrência |
| `crm_cd_desc` | string | 100% | Tipo de crime (~140 categorias) |
| `vict_age` | int64 | 100%\* | Idade da vítima (`0` = sem informação) |
| `vict_sex` | string | ~86% | M / F / X / H / NaN |
| `weapon_used_cd` | float | ~33% | Código da arma (NaN = sem arma declarada) |
| `lat` | float64 | **100%** | Latitude individual de cada ocorrência |
| `lon` | float64 | **100%** | Longitude individual de cada ocorrência |
| `area_name` | string | 100% | 21 divisões operacionais do LAPD |

### Principais achados

**Coordenadas:** 100% dos registros têm lat/lon individuais, sem nulos e sem zeros. O heatmap do Flutter pode usar pontos geográficos reais diretamente.

**Padrão temporal:** pico entre **12h–18h**, mais forte na sexta e sábado. O padrão é estável entre 2020 e 2025 — base confiável para o risk_score.

**Tipos:** furto de veículo, roubo a pedestres e assalto simples dominam. Crimes com arma são ~33% do total.

**Áreas críticas:** as 5 áreas com maior volume recebem heatmap em vermelho no app.

### Regras para `calculate_risk_score.dart`
- Hora 12h–18h → peso elevado
- Sex/Sáb → ajuste marginal positivo
- `weapon_used_cd not null` → multiplicador de gravidade
- Top 5 áreas LAPD → floor mínimo de risco moderado
- `weapon_used_cd` nulo = crime sem arma, não dado faltante → binarizar como `has_weapon`